# AHỤIKE 02: QLoRA fine-tuning of N-ATLaS-8B (Unsloth, ~3.5 h)
Trains the **parallel-anchored variant** (each case in all four languages) and **AHỤIKE**, the released model (same size and language mix, more distinct cases, each in one language). 1 epoch each, ~1.5 h on Kaggle T4s. Each adapter is pushed to the private work repo as soon as it finishes.

Kaggle settings: **GPU T4 x2**, **Internet on**, Secret **HF_TOKEN** ticked for this notebook. Run with **Save Version > Save & Run All (Commit)**.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')

In [ ]:
!git clone -q https://github.com/EgwuSamuel/ahuike.git /kaggle/working/ahuike || (cd /kaggle/working/ahuike && git pull -q)
%cd /kaggle/working/ahuike
!pip install -q unsloth
!pip uninstall -y -q torchaudio
!python scripts/check_setup.py
!python scripts/hub_sync.py pull data
!python scripts/build_sft.py

## 1. Parallel-anchored variant

In [ ]:
!python scripts/finetune.py --data data/sft_anchored.jsonl --out outputs/ahuike-lora
!python scripts/hub_sync.py push outputs/ahuike-lora

## 2. AHỤIKE: diverse cases (the released model; adapter folder name kept from the experiment)

In [ ]:
!python scripts/finetune.py --data data/sft_ablation.jsonl --out outputs/ablation-lora
!python scripts/hub_sync.py push outputs/ablation-lora

## 3. (Optional) merged Q4_K_M GGUF for the CPU demo
The repo name must carry the **Powered-by-Awarri** suffix (N-ATLaS licence).

In [ ]:
HF_USER = 'SamEgwu'
!python scripts/export_gguf.py --adapter outputs/ablation-lora --out /tmp/ahuike-gguf --push {HF_USER}/AHUIKE-N-ATLaS-8B-GGUF-Powered-by-Awarri